In [ ]:
!pip install transformers datasets torch scikit-learn accelerate

In [ ]:
import pandas as pd

df = pd.read_csv("/content/go_emotions_dataset.csv")

print(df.head())
print("\nDataset Shape:", df.shape)
print("\nColumns:")
print(df.columns)

In [ ]:
# 1. Missing values per column
print("Missing values per column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

# 2. Duplicate checks
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate ids:", df['id'].duplicated().sum())
print("Duplicate texts:", df['text'].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes.value_counts())

# 4. Empty or very short texts
print("\nEmpty texts:", (df['text'].str.strip() == '').sum())

In [ ]:
# All emotion columns (everything except id, text, example_very_unclear)
emotion_cols = [c for c in df.columns if c not in ['id', 'text', 'example_very_unclear']]

# 1. Count of rows labeled with each emotion
emotion_counts = df[emotion_cols].sum().sort_values(ascending=False)
print("Emotion counts (all rows):")
print(emotion_counts)

# 2. Focus on your six target emotions
target = ['joy', 'sadness', 'love', 'anger', 'fear', 'surprise']
print("\nTarget emotion counts:")
print(df[target].sum())

# 3. How many labels does each row have?
print("\nLabels per row:")
print(df[emotion_cols].sum(axis=1).value_counts().sort_index())

# 4. Unclear examples
print("\nVery unclear examples:", df['example_very_unclear'].sum())

# 5. Bar chart
import matplotlib.pyplot as plt
emotion_counts.plot(kind='bar', figsize=(14, 5), title='Emotion Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
target = ['joy', 'sadness', 'love', 'anger', 'fear', 'surprise']
other_cols = [c for c in emotion_cols if c not in target]

# 1. Remove unclear examples
d = df[df['example_very_unclear'] == False]

# 2. Keep rows with exactly one target emotion and no other emotion
mask = (d[target].sum(axis=1) == 1) & (d[other_cols].sum(axis=1) == 0)
sub = d[mask].copy()

# 3. Create a single text label column
sub['label'] = sub[target].idxmax(axis=1)

# 4. Remove comments where raters gave conflicting target labels
n_labels = sub.groupby('id')['label'].nunique()
conflict_ids = n_labels[n_labels > 1].index
sub = sub[~sub['id'].isin(conflict_ids)]

# 5. One row per comment
sub = sub.drop_duplicates(subset='id')

# Keep only what we need
sub = sub[['id', 'text', 'label']].reset_index(drop=True)

print("Conflicting comments removed:", len(conflict_ids))
print("Final shape:", sub.shape)
print("\nClass counts:")
print(sub['label'].value_counts())
print("\nClass percentages:")
print((sub['label'].value_counts(normalize=True) * 100).round(1))

print("\nSample rows:")
print(sub.sample(8, random_state=42))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 1. Label <-> id mappings
labels = sorted(sub['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
sub['label_id'] = sub['label'].map(label2id)
print("label2id:", label2id)

# 2. Stratified split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    sub, test_size=0.20, stratify=sub['label_id'], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label_id'], random_state=42)

print("\nSizes -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

# 3. Check class proportions are preserved
print("\nClass % per split:")
print(pd.DataFrame({
    'train': train_df['label'].value_counts(normalize=True),
    'val':   val_df['label'].value_counts(normalize=True),
    'test':  test_df['label'].value_counts(normalize=True),
}).mul(100).round(1))

# 4. Class weights (computed from the training set only)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(labels)),
    y=train_df['label_id'])
print("\nClass weights:")
for l, w in zip(labels, weights):
    print(f"  {l}: {w:.2f}")

# 5. Make sure no comment appears in more than one split
print("\nOverlap train/test:", len(set(train_df['id']) & set(test_df['id'])))
print("Overlap train/val:", len(set(train_df['id']) & set(val_df['id'])))

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 1. Word counts (quick view)
word_lens = train_df['text'].str.split().str.len()
print("Word count stats:")
print(word_lens.describe())

# 2. Token counts (what DistilBERT actually sees)
token_lens = train_df['text'].apply(lambda t: len(tokenizer.encode(t, truncation=False)))
print("\nToken count stats:")
print(token_lens.describe())

# 3. Percentiles to choose max_length
for p in [90, 95, 99, 99.5]:
    print(f"{p}th percentile: {int(np.percentile(token_lens, p))} tokens")

# 4. How much would be cut off at different max_length values?
for m in [32, 48, 64, 128]:
    pct = (token_lens > m).mean() * 100
    print(f"max_length={m}: {pct:.2f}% of texts truncated")

# 5. Example of tokenization
example = train_df['text'].iloc[0]
print("\nExample text:", example)
print("Tokens:", tokenizer.tokenize(example))

# 6. Histogram
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.hist(token_lens, bins=40)
plt.title("Token length distribution (train)")
plt.xlabel("Tokens")
plt.ylabel("Number of comments")
plt.show()

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import DataCollatorWithPadding

MAX_LEN = 48

# 1. Convert pandas DataFrames to HF Datasets (keep text + numeric label only)
def to_hf(df_):
    d = df_[['text', 'label_id']].rename(columns={'label_id': 'labels'}).reset_index(drop=True)
    return Dataset.from_pandas(d)

raw_ds = DatasetDict({
    'train': to_hf(train_df),
    'validation': to_hf(val_df),
    'test': to_hf(test_df),
})

# 2. Tokenize (no padding here; the collator pads per batch)
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

# 3. Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Inspect the result
print(tokenized_ds)
print("\nColumns:", tokenized_ds['train'].column_names)
sample = tokenized_ds['train'][0]
print("\nSample input_ids:", sample['input_ids'])
print("Sample label:", sample['labels'], "->", id2label[sample['labels']])
print("Decoded:", tokenizer.decode(sample['input_ids']))

# 5. Test the collator on a small batch
batch = data_collator([tokenized_ds['train'][i] for i in range(4)])
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {tuple(v.shape)}")

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

print("Evaluate library installed successfully!")

In [ ]:
import torch
import torch.nn as nn
import evaluate

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y, preds),
        "macro_f1": f1_score(y, preds, average="macro"),
    }

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully!")
print("Number of classes:", model.config.num_labels)
print("Emotion labels:", id2label)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./moodmentor_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

print("Training arguments configured successfully!")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized successfully!")

In [ ]:
training_results = trainer.train()

print("MoodMentor training completed successfully!")

In [ ]:
trainer.save_model("./moodmentor_emotion_model")

tokenizer.save_pretrained(
    "./moodmentor_emotion_model"
)

print("MoodMentor model saved successfully!")

In [ ]:
test_results = trainer.evaluate(
    tokenized_ds["test"]
)

print("Final Test Results:")
print(test_results)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predict on the test dataset
predictions = trainer.predict(tokenized_ds["test"])

# Get predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

# Get actual labels
y_true = predictions.label_ids

# Generate classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=labels,
    digits=4
)

print("MoodMentor Classification Report:")
print(report)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(labels)))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    cmap="Blues",
    values_format="d",
    ax=ax
)

plt.title("MoodMentor Emotion Classification")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
trainer.save_model("./moodmentor_emotion_model")

tokenizer.save_pretrained(
    "./moodmentor_emotion_model"
)

print("MoodMentor model saved successfully!")

In [ ]:
import torch

def predict_emotion(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits, dim=-1
    )

    predicted_id = torch.argmax(
        probabilities, dim=-1
    ).item()

    emotion = id2label[predicted_id]

    confidence = probabilities[0][predicted_id].item()

    print("Input text:", text)
    print("Predicted emotion:", emotion)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 40)


predict_emotion("I am extremely happy today!")
predict_emotion("I feel very lonely and sad.")
predict_emotion("I am scared about my future.")
predict_emotion("I love spending time with my family.")
predict_emotion("I am so angry with this situation.")
predict_emotion("Wow! I did not expect this surprise!")

In [ ]:
import shutil

shutil.make_archive(
    "/content/moodmentor_emotion_model",
    "zip",
    "/content/moodmentor_emotion_model"
)

print("Model ZIP file created successfully!")

In [ ]:
from google.colab import files

files.download(
    "/content/moodmentor_emotion_model.zip"
)

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import classification_report
import numpy as np

BERT_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

def bert_tokenize_fn(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

bert_ds = raw_ds.map(bert_tokenize_fn, batched=True, remove_columns=["text"])
bert_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)

bert_args = TrainingArguments(
    output_dir="./moodmentor_bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    seed=42,
    report_to="none",
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_ds["train"],
    eval_dataset=bert_ds["validation"],
    processing_class=bert_tokenizer,
    data_collator=bert_collator,
    compute_metrics=compute_metrics,
)

bert_trainer.train()

bert_test = bert_trainer.evaluate(bert_ds["test"])
print("BERT test results:", bert_test)

bert_pred = bert_trainer.predict(bert_ds["test"])
print(classification_report(
    bert_pred.label_ids,
    np.argmax(bert_pred.predictions, axis=1),
    target_names=labels,
    digits=4,
))

bert_trainer.save_model("./moodmentor_bert_model")
bert_tokenizer.save_pretrained("./moodmentor_bert_model")
print("BERT model saved.")

In [ ]:
import os
print(os.path.exists("/content/go_emotions_dataset.csv"))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset, DatasetDict

df = pd.read_csv("/content/go_emotions_dataset.csv")

emotion_cols = [c for c in df.columns if c not in ['id', 'text', 'example_very_unclear']]
target = ['joy', 'sadness', 'love', 'anger', 'fear', 'surprise']
other_cols = [c for c in emotion_cols if c not in target]

d = df[df['example_very_unclear'] == False]
mask = (d[target].sum(axis=1) == 1) & (d[other_cols].sum(axis=1) == 0)
sub = d[mask].copy()
sub['label'] = sub[target].idxmax(axis=1)

n_labels = sub.groupby('id')['label'].nunique()
conflict_ids = n_labels[n_labels > 1].index
sub = sub[~sub['id'].isin(conflict_ids)]
sub = sub.drop_duplicates(subset='id')
sub = sub[['id', 'text', 'label']].reset_index(drop=True)

labels = sorted(sub['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
sub['label_id'] = sub['label'].map(label2id)

train_df, temp_df = train_test_split(
    sub, test_size=0.20, stratify=sub['label_id'], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label_id'], random_state=42)

MAX_LEN = 48

def to_hf(df_):
    d_ = df_[['text', 'label_id']].rename(columns={'label_id': 'labels'}).reset_index(drop=True)
    return Dataset.from_pandas(d_)

raw_ds = DatasetDict({
    'train': to_hf(train_df),
    'validation': to_hf(val_df),
    'test': to_hf(test_df),
})

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y, preds),
        "macro_f1": f1_score(y, preds, average="macro"),
    }

print("Labels:", labels)
print("Sizes -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip -o -q moodmentor_colab_bundle.zip -d /content/moodmentor

import sys
import pandas as pd
sys.path.insert(0, "/content/moodmentor")

from ml.data_prep import labels

base = "/content/moodmentor/ml/data/processed/goemotions"
for name in ["train", "validation_clean", "test_clean"]:
    d = pd.read_parquet(f"{base}/{name}.parquet")
    print(name, len(d), d.columns.tolist())

print("Number of labels:", labels.NUM_LABELS)
print("First 5:", labels.LABELS[:5], "| last:", labels.LABELS[-1])
print("Ekman groups:", labels.EKMAN_GROUPS)

In [ ]:
import math
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score

# 1. Environment & Config Setup
sys.path.insert(0, "/content/moodmentor")
from ml.data_prep import labels as L

SEED = 42
OUT_NAME = "bert_goemotions"
MAX_LEN = 64
BASE = "/content/moodmentor/ml/data/processed/goemotions"

# 2. Load Split Data
def load_split(name):
    df = pd.read_parquet(f"{BASE}/{name}.parquet")
    y = L.multi_hot([list(x) for x in df["labels"]]).astype("float32")
    return df["text"].tolist(), y

train_text, y_train = load_split("train")
val_text, y_val = load_split("validation_clean")
test_text, y_test = load_split("test_clean")

# 3. Initialize Tokenizer & Datasets
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def make_ds(texts, y):
    ds = Dataset.from_dict({"text": texts, "labels": y.tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds = make_ds(train_text, y_train)
val_ds = make_ds(val_text, y_val)
test_ds = make_ds(test_text, y_test)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Model Setup
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=L.NUM_LABELS, problem_type="multi_label_classification",
    id2label={i: n for i, n in enumerate(L.LABELS)},
    label2id={n: i for i, n in enumerate(L.LABELS)},
)

# 5. Class Weights & Custom Loss Trainer
pos = y_train.sum(axis=0)
neg = len(y_train) - pos
pos_weight = np.minimum(np.sqrt(neg / np.maximum(pos, 1)), 10.0)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)

class WeightedBCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t.to(logits.device))(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = (sigmoid(logits) >= 0.5).astype(int)
    y = y.astype(int)
    return {
        "macro_f1": f1_score(y, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(y, preds, average="micro", zero_division=0)
    }

# 6. Training Arguments & Execution
steps_per_epoch = math.ceil(len(train_ds) / 32)
total_steps = steps_per_epoch * 4
warmup = int(0.1 * total_steps)
print(f"steps/epoch: {steps_per_epoch} | total: {total_steps} | warmup steps: {warmup}")

args = TrainingArguments(
    output_dir=f"./{OUT_NAME}_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=warmup,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = WeightedBCETrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training finished.")

In [ ]:
import json, os

# 1) Pick ONE global threshold using the validation set only
val_prob = sigmoid(trainer.predict(val_ds).predictions)
y_val_i = y_val.astype(int)

best_t, best_f1 = 0.5, -1.0
for t in np.arange(0.10, 0.91, 0.05):
    f = f1_score(y_val_i, (val_prob >= t).astype(int), average="macro", zero_division=0)
    print(f"threshold {t:.2f}: validation macro F1 = {f:.4f}")
    if f > best_f1:
        best_t, best_f1 = float(round(t, 2)), float(f)
print(f"\nChosen threshold: {best_t} (validation macro F1 {best_f1:.4f})")

# 2) Score the test set once, with that threshold
test_prob = sigmoid(trainer.predict(test_ds).predictions)
y_test_i = y_test.astype(int)
test_pred = (test_prob >= best_t).astype(int)

def summary(name, yt, yp):
    return {
        "view": name,
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(yt, yp, average="micro", zero_division=0)),
    }

results = {
    "model": MODEL_NAME, "threshold": best_t, "validation_macro_f1": best_f1,
    "views": [
        summary("28 labels", y_test_i, test_pred),
        summary("Ekman 6 + neutral", L.to_ekman_view(y_test_i), L.to_ekman_view(test_pred)),
        summary("12 headline emotions", L.to_headline_view(y_test_i), L.to_headline_view(test_pred)),
    ],
}
for v in results["views"]:
    print(f"{v['view']:<24} macro F1 {v['macro_f1']:.4f} | micro F1 {v['micro_f1']:.4f}")

# 3) Per-label F1 (all 28)
per_f1 = f1_score(y_test_i, test_pred, average=None, zero_division=0)
support = y_test_i.sum(axis=0)
print("\nPer-label test F1:")
for name, f, s in sorted(zip(L.LABELS, per_f1, support), key=lambda r: -r[1]):
    print(f"  {name:<15} F1 {f:.3f}  (n={int(s)})")
results["per_label_f1"] = {n: float(f) for n, f in zip(L.LABELS, per_f1)}

# 4) Save everything (model stays in Colab for now; results come to VS Code)
os.makedirs("/content/results", exist_ok=True)
np.save(f"/content/results/{OUT_NAME}_test_prob.npy", test_prob)
json.dump(results, open(f"/content/results/{OUT_NAME}_results.json", "w"), indent=2)
trainer.save_model(f"/content/{OUT_NAME}_model")
tokenizer.save_pretrained(f"/content/{OUT_NAME}_model")

import shutil
from google.colab import files
shutil.make_archive("/content/results_distilbert", "zip", "/content/results")
files.download("/content/results_distilbert.zip")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/MoodMentor
!cp -r /content/distilbert_goemotions_model /content/drive/MyDrive/MoodMentor/
!ls /content/drive/MyDrive/MoodMentor/distilbert_goemotions_model

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/distilbert_goemotions_model", "zip", "/content/distilbert_goemotions_model")
files.download("/content/distilbert_goemotions_model.zip")

In [ ]:
import json, os, shutil
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)
from ml.data_prep import labels as L
from google.colab import files

MODELS = {
    "distilbert": "/content/distilbert_goemotions_model",
    "bert": "/content/bert_goemotions_model",
}
PROC = "/content/moodmentor/ml/data/processed"
MAX_LEN = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

POSITIVE = ["admiration", "amusement", "approval", "caring", "desire", "excitement",
            "gratitude", "joy", "love", "optimism", "pride", "relief"]
NEGATIVE = ["anger", "annoyance", "disappointment", "disapproval", "disgust",
            "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"]
pos_idx = [L.LABEL_INDEX[n] for n in POSITIVE]
neg_idx = [L.LABEL_INDEX[n] for n in NEGATIVE]
ISEAR_MAIN = ["joy", "fear", "anger", "sadness", "disgust"]

semeval = pd.read_parquet(f"{PROC}/semeval2018_voc_binary/test.parquet")
isear = pd.read_parquet(f"{PROC}/isear/isear_clean.parquet")
isear_main = isear[isear["in_main_eval"]].reset_index(drop=True)
isear_other = isear[~isear["in_main_eval"]].reset_index(drop=True)
print("SemEval test:", len(semeval), "| ISEAR main:", len(isear_main), "| ISEAR shame/guilt:", len(isear_other))

def predict_probs(model, tok, texts, batch_size=64):
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        out.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.vstack(out)

os.makedirs("/content/results_eval", exist_ok=True)

for name, path in MODELS.items():
    if not os.path.isdir(path):
        print(f"\n[{name}] folder not found at {path}, skipped")
        continue
    print(f"\n===== {name} =====")
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device)

    # --- SemEval polarity ---
    p = predict_probs(model, tok, semeval["text"].tolist())
    y_true = semeval["label_id"].to_numpy()
    y_pred = (p[:, pos_idx].max(axis=1) > p[:, neg_idx].max(axis=1)).astype(int)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    sem = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "per_class": {n: {"precision": float(prec[i]), "recall": float(rec[i]),
                          "f1": float(f1[i]), "support": int(sup[i])}
                      for i, n in enumerate(["negative", "positive"])},
        "confusion_matrix_rows_true_cols_pred_[neg,pos]": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }
    print(f"SemEval polarity: accuracy {sem['accuracy']:.4f} | macro F1 {sem['macro_f1']:.4f}")
    print("  confusion [neg,pos]:", sem["confusion_matrix_rows_true_cols_pred_[neg,pos]"])
    semeval.assign(pred=y_pred)[["id", "text", "label_id", "pred"]].to_csv(
        f"/content/results_eval/{name}_semeval_predictions.csv", index=False)

    # --- ISEAR main rows (single predicted label = top-1 of 28) ---
    p = predict_probs(model, tok, isear_main["text"].tolist())
    pred_names = np.array([L.LABELS[i] for i in p.argmax(axis=1)])
    true_names = isear_main["mapped_label"].to_numpy()
    prec, rec, f1, sup = precision_recall_fscore_support(
        true_names, pred_names, labels=ISEAR_MAIN, zero_division=0)
    isr = {
        "n": int(len(isear_main)),
        "accuracy": float((pred_names == true_names).mean()),
        "macro_f1_5_classes": float(f1_score(true_names, pred_names, labels=ISEAR_MAIN,
                                             average="macro", zero_division=0)),
        "predicted_outside_5_classes_share": float((~np.isin(pred_names, ISEAR_MAIN)).mean()),
        "per_class": {n: {"precision": float(prec[i]), "recall": float(rec[i]),
                          "f1": float(f1[i]), "support": int(sup[i])}
                      for i, n in enumerate(ISEAR_MAIN)},
    }
    print(f"ISEAR main: accuracy {isr['accuracy']:.4f} | macro F1 {isr['macro_f1_5_classes']:.4f} "
          f"| predicted outside the 5 classes: {isr['predicted_outside_5_classes_share']:.1%}")
    for n, v in isr["per_class"].items():
        print(f"  {n:<9} P {v['precision']:.3f}  R {v['recall']:.3f}  F1 {v['f1']:.3f}  (n={v['support']})")
    isear_main.assign(pred=pred_names)[["id", "text", "emotion_original", "pred"]].to_csv(
        f"/content/results_eval/{name}_isear_predictions.csv", index=False)

    # --- ISEAR shame/guilt: only the distribution of predictions ---
    p = predict_probs(model, tok, isear_other["text"].tolist())
    other_names = np.array([L.LABELS[i] for i in p.argmax(axis=1)])
    dist = {}
    for emo in ["shame", "guilt"]:
        vc = pd.Series(other_names[(isear_other["emotion_original"] == emo).to_numpy()]).value_counts()
        dist[emo] = {k: int(v) for k, v in vc.head(6).items()}
        print(f"  {emo} rows, top predicted labels:", dist[emo])

    json.dump({"model": name, "semeval_polarity": sem, "isear_main": isr,
               "isear_shame_guilt_top_predictions": dist},
              open(f"/content/results_eval/{name}_eval.json", "w"), indent=2)

    del model
    torch.cuda.empty_cache()

shutil.make_archive("/content/results_eval", "zip", "/content/results_eval")
files.download("/content/results_eval.zip")

In [ ]:
MODELS = {"bert": "/content/bert_goemotions_model"}

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/distilbert_goemotions_model", "zip", "/content/distilbert_goemotions_model")
files.download("/content/distilbert_goemotions_model.zip")

In [ ]:
import os, sys, json, math, shutil
import numpy as np, pandas as pd, torch, torch.nn as nn
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import f1_score
from google.colab import files

if not os.path.isdir("/content/moodmentor/ml"):
    raise FileNotFoundError("Data bundle missing. Upload moodmentor_colab_bundle.zip, then run: "
                            "!unzip -o -q moodmentor_colab_bundle.zip -d /content/moodmentor")
sys.path.insert(0, "/content/moodmentor")
from ml.data_prep import labels as L

SEED = 42
set_seed(SEED)
MODEL_NAME = "bert-base-uncased"
OUT_NAME = "bert_goemotions"
MAX_LEN = 64
BASE = "/content/moodmentor/ml/data/processed/goemotions"
print("GPU available:", torch.cuda.is_available())

def load_split(name):
    df = pd.read_parquet(f"{BASE}/{name}.parquet")
    y = L.multi_hot([list(x) for x in df["labels"]]).astype("float32")
    return df["text"].tolist(), y

train_text, y_train = load_split("train")
val_text, y_val = load_split("validation_clean")
test_text, y_test = load_split("test_clean")
print("sizes:", len(train_text), len(val_text), len(test_text))

pos = y_train.sum(axis=0)
neg = len(y_train) - pos
pos_weight = np.minimum(np.sqrt(neg / np.maximum(pos, 1)), 10.0)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(texts, y):
    ds = Dataset.from_dict({"text": texts, "labels": y.tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = make_ds(train_text, y_train), make_ds(val_text, y_val), make_ds(test_text, y_test)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=L.NUM_LABELS, problem_type="multi_label_classification",
    id2label={i: n for i, n in enumerate(L.LABELS)},
    label2id={n: i for i, n in enumerate(L.LABELS)},
)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = (sigmoid(logits) >= 0.5).astype(int)
    y = y.astype(int)
    return {"macro_f1": f1_score(y, preds, average="macro", zero_division=0),
            "micro_f1": f1_score(y, preds, average="micro", zero_division=0)}

class WeightedBCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t.to(logits.device))(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

steps_per_epoch = math.ceil(len(train_ds) / 32)
warmup = int(0.1 * steps_per_epoch * 4)

args = TrainingArguments(
    output_dir=f"./{OUT_NAME}_checkpoints", eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=2, learning_rate=5e-5, per_device_train_batch_size=32,
    per_device_eval_batch_size=64, num_train_epochs=4, weight_decay=0.01,
    warmup_steps=warmup, logging_steps=200, load_best_model_at_end=True,
    metric_for_best_model="macro_f1", greater_is_better=True,
    fp16=torch.cuda.is_available(), seed=SEED, report_to="none",
)

trainer = WeightedBCETrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                             processing_class=tokenizer, data_collator=collator,
                             compute_metrics=compute_metrics)
trainer.train()
print("BERT training finished.")

# Threshold tuned on VALIDATION only
val_prob = sigmoid(trainer.predict(val_ds).predictions)
y_val_i = y_val.astype(int)
best_t, best_f1 = 0.5, -1.0
for t in np.arange(0.10, 0.91, 0.05):
    f = f1_score(y_val_i, (val_prob >= t).astype(int), average="macro", zero_division=0)
    print(f"threshold {t:.2f}: validation macro F1 = {f:.4f}")
    if f > best_f1:
        best_t, best_f1 = float(round(t, 2)), float(f)
print(f"\nChosen threshold: {best_t} (validation macro F1 {best_f1:.4f})")

# Test set scored once
test_prob = sigmoid(trainer.predict(test_ds).predictions)
y_test_i = y_test.astype(int)
test_pred = (test_prob >= best_t).astype(int)

def summary(name, yt, yp):
    return {"view": name,
            "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
            "micro_f1": float(f1_score(yt, yp, average="micro", zero_division=0))}

results = {"model": MODEL_NAME, "threshold": best_t, "validation_macro_f1": best_f1,
           "views": [summary("28 labels", y_test_i, test_pred),
                     summary("Ekman 6 + neutral", L.to_ekman_view(y_test_i), L.to_ekman_view(test_pred)),
                     summary("12 headline emotions", L.to_headline_view(y_test_i), L.to_headline_view(test_pred))]}
for v in results["views"]:
    print(f"{v['view']:<24} macro F1 {v['macro_f1']:.4f} | micro F1 {v['micro_f1']:.4f}")

per_f1 = f1_score(y_test_i, test_pred, average=None, zero_division=0)
support = y_test_i.sum(axis=0)
print("\nPer-label test F1:")
for name, f, s in sorted(zip(L.LABELS, per_f1, support), key=lambda r: -r[1]):
    print(f"  {name:<15} F1 {f:.3f}  (n={int(s)})")
results["per_label_f1"] = {n: float(f) for n, f in zip(L.LABELS, per_f1)}

os.makedirs("/content/results_bert", exist_ok=True)
np.save(f"/content/results_bert/{OUT_NAME}_test_prob.npy", test_prob)
json.dump(results, open(f"/content/results_bert/{OUT_NAME}_results.json", "w"), indent=2)
trainer.save_model(f"/content/{OUT_NAME}_model")
tokenizer.save_pretrained(f"/content/{OUT_NAME}_model")
shutil.make_archive("/content/results_bert", "zip", "/content/results_bert")
files.download("/content/results_bert.zip")

In [ ]:
MODELS = {"bert": "/content/bert_goemotions_model"}

In [ ]:
import os
print("BERT model folder:", os.path.isdir("/content/bert_goemotions_model"))
print("Eval folder:", os.path.isdir("/content/results_eval"))
!ls /content/results_eval

In [ ]:
MODELS = {"bert": "/content/bert_goemotions_model"}

In [ ]:
import os
print("BERT model folder:", os.path.isdir("/content/bert_goemotions_model"))
print("DistilBERT model folder:", os.path.isdir("/content/distilbert_goemotions_model"))
print("Bundle unzipped:", os.path.isdir("/content/moodmentor/ml"))

In [ ]:
MODELS = {"bert": "/content/bert_goemotions_model"}

In [ ]:
import json, os, shutil
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)
import sys
sys.path.insert(0, "/content/moodmentor")
from ml.data_prep import labels as L
from google.colab import files

MODELS = {"bert": "/content/bert_goemotions_model"}
PROC = "/content/moodmentor/ml/data/processed"
MAX_LEN = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

POSITIVE = ["admiration", "amusement", "approval", "caring", "desire", "excitement",
            "gratitude", "joy", "love", "optimism", "pride", "relief"]
NEGATIVE = ["anger", "annoyance", "disappointment", "disapproval", "disgust",
            "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"]
pos_idx = [L.LABEL_INDEX[n] for n in POSITIVE]
neg_idx = [L.LABEL_INDEX[n] for n in NEGATIVE]
ISEAR_MAIN = ["joy", "fear", "anger", "sadness", "disgust"]

semeval = pd.read_parquet(f"{PROC}/semeval2018_voc_binary/test.parquet")
isear = pd.read_parquet(f"{PROC}/isear/isear_clean.parquet")
isear_main = isear[isear["in_main_eval"]].reset_index(drop=True)
isear_other = isear[~isear["in_main_eval"]].reset_index(drop=True)
print("SemEval test:", len(semeval), "| ISEAR main:", len(isear_main), "| ISEAR shame/guilt:", len(isear_other))

def predict_probs(model, tok, texts, batch_size=64):
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        out.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.vstack(out)

os.makedirs("/content/results_eval", exist_ok=True)

for name, path in MODELS.items():
    if not os.path.isdir(path):
        print(f"\n[{name}] folder not found at {path}, skipped")
        continue
    print(f"\n===== {name} =====")
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device)

    # Reload check: must reproduce the probabilities saved at training time
    saved = f"/content/results_{name}/{name}_goemotions_test_prob.npy"
    if os.path.exists(saved):
        test_df = pd.read_parquet(f"{PROC}/goemotions/test_clean.parquet")
        chk = predict_probs(model, tok, test_df["text"].tolist())
        diff = float(np.abs(chk - np.load(saved)).max())
        print(f"Reload check: max difference vs saved test probabilities = {diff:.4f}")

    # --- SemEval polarity ---
    p = predict_probs(model, tok, semeval["text"].tolist())
    y_true = semeval["label_id"].to_numpy()
    y_pred = (p[:, pos_idx].max(axis=1) > p[:, neg_idx].max(axis=1)).astype(int)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    sem = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "per_class": {n: {"precision": float(prec[i]), "recall": float(rec[i]),
                          "f1": float(f1[i]), "support": int(sup[i])}
                      for i, n in enumerate(["negative", "positive"])},
        "confusion_matrix_rows_true_cols_pred_[neg,pos]": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }
    print(f"SemEval polarity: accuracy {sem['accuracy']:.4f} | macro F1 {sem['macro_f1']:.4f}")
    print("  confusion [neg,pos]:", sem["confusion_matrix_rows_true_cols_pred_[neg,pos]"])
    semeval.assign(pred=y_pred)[["id", "text", "label_id", "pred"]].to_csv(
        f"/content/results_eval/{name}_semeval_predictions.csv", index=False)

    # --- ISEAR main rows (single predicted label = top-1 of 28) ---
    p = predict_probs(model, tok, isear_main["text"].tolist())
    pred_names = np.array([L.LABELS[i] for i in p.argmax(axis=1)])
    true_names = isear_main["mapped_label"].to_numpy()
    prec, rec, f1, sup = precision_recall_fscore_support(
        true_names, pred_names, labels=ISEAR_MAIN, zero_division=0)
    isr = {
        "n": int(len(isear_main)),
        "accuracy": float((pred_names == true_names).mean()),
        "macro_f1_5_classes": float(f1_score(true_names, pred_names, labels=ISEAR_MAIN,
                                             average="macro", zero_division=0)),
        "predicted_outside_5_classes_share": float((~np.isin(pred_names, ISEAR_MAIN)).mean()),
        "per_class": {n: {"precision": float(prec[i]), "recall": float(rec[i]),
                          "f1": float(f1[i]), "support": int(sup[i])}
                      for i, n in enumerate(ISEAR_MAIN)},
    }
    print(f"ISEAR main: accuracy {isr['accuracy']:.4f} | macro F1 {isr['macro_f1_5_classes']:.4f} "
          f"| predicted outside the 5 classes: {isr['predicted_outside_5_classes_share']:.1%}")
    for n, v in isr["per_class"].items():
        print(f"  {n:<9} P {v['precision']:.3f}  R {v['recall']:.3f}  F1 {v['f1']:.3f}  (n={v['support']})")
    isear_main.assign(pred=pred_names)[["id", "text", "emotion_original", "pred"]].to_csv(
        f"/content/results_eval/{name}_isear_predictions.csv", index=False)

    # --- ISEAR shame/guilt: distribution of predictions only ---
    p = predict_probs(model, tok, isear_other["text"].tolist())
    other_names = np.array([L.LABELS[i] for i in p.argmax(axis=1)])
    dist = {}
    for emo in ["shame", "guilt"]:
        vc = pd.Series(other_names[(isear_other["emotion_original"] == emo).to_numpy()]).value_counts()
        dist[emo] = {k: int(v) for k, v in vc.head(6).items()}
        print(f"  {emo} rows, top predicted labels:", dist[emo])

    json.dump({"model": name, "semeval_polarity": sem, "isear_main": isr,
               "isear_shame_guilt_top_predictions": dist},
              open(f"/content/results_eval/{name}_eval.json", "w"), indent=2)

    del model
    torch.cuda.empty_cache()

shutil.make_archive("/content/results_eval", "zip", "/content/results_eval")
files.download("/content/results_eval.zip")

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print("Logged in as:", whoami()["name"])

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/bert_goemotions_model", "zip", "/content/bert_goemotions_model")
files.download("/content/bert_goemotions_model.zip")

In [ ]:
import json, os
from huggingface_hub import HfApi, whoami

try:
    USER = whoami()["name"]
except Exception:
    raise SystemExit("Not logged in. Finish the HF_TOKEN / login step first.")

MODEL_DIR = "/content/bert_goemotions_model"
REPO_ID = f"{USER}/moodmentor-bert-goemotions"
assert os.path.isdir(MODEL_DIR), "BERT model folder not found"

res = json.load(open("/content/results_bert/bert_goemotions_results.json"))
views = {v["view"]: v for v in res["views"]}
THRESHOLD = res["threshold"]

def f(x):
    return f"{x:.4f}"

ext = "External evaluation results were not recorded when this card was written."
if os.path.exists("/content/results_eval/bert_eval.json"):
    e = json.load(open("/content/results_eval/bert_eval.json"))
    s, i = e["semeval_polarity"], e["isear_main"]
    ext = (
        f"- SemEval-2018 V-oc binary polarity (675 test tweets, neutral excluded): "
        f"accuracy {s['accuracy']:.3f}, macro F1 {s['macro_f1']:.3f}. "
        f"A simple VADER baseline scored about 0.837 accuracy, so the model is not better than the baseline here.\n"
        f"- ISEAR held-out (3,752 rows, 5 directly mapped emotions, top-1 of 28 labels): "
        f"accuracy {i['accuracy']:.3f}, macro F1 {i['macro_f1_5_classes']:.3f}. "
        f"{i['predicted_outside_5_classes_share']:.0%} of predictions fell outside the 5 ISEAR classes "
        f"(mostly neutral), which shows a large domain shift."
    )

with open(f"{MODEL_DIR}/moodmentor_config.json", "w") as fh:
    json.dump({"threshold": THRESHOLD, "max_length": 64,
               "note": "Global threshold chosen on the validation set."}, fh, indent=2)

card = """---
license: apache-2.0
language: en
base_model: bert-base-uncased
datasets:
- google-research-datasets/go_emotions
pipeline_tag: text-classification
tags:
- text-classification
- multi-label-classification
- emotion
- goemotions
---

# MoodMentor BERT (GoEmotions, 28 labels, multi-label)

A `bert-base-uncased` model fine-tuned for multi-label emotion detection on a cleaned version of GoEmotions (English Reddit comments, 28 labels). Built for a college project (MoodMentor). **It is not a medical or psychological tool and must not be used to diagnose anyone.**

## Intended use
- Showing which emotions a short English text may express, with a confidence score for each.
- Educational and demonstration use.

## Not intended for
- Diagnosis, therapy decisions, or any high-stakes decision about a person.
- Monitoring people without their knowledge or consent.
- Non-English text, or very different domains without re-evaluation.

## Training
- Data: GoEmotions, simplified config, cleaned train/validation/test files (unclear examples removed from evaluation sets).
- Settings: max length 64, learning rate 5e-5, batch size 32, 4 epochs, weight decay 0.01, seed 42.
- Loss: binary cross-entropy with per-label positive weights (square root of negatives/positives, capped at 10).
- Best epoch chosen on validation macro F1. One global decision threshold (__THRESHOLD__) chosen on the validation set only.

## Results (cleaned GoEmotions test set, one run, seed 42)
| View | Macro F1 | Micro F1 |
|---|---|---|
| 28 labels | __M28__ | __U28__ |
| Ekman 6 + neutral | __MEK__ | __UEK__ |
| 12 headline emotions | __MHL__ | __UHL__ |

Weak labels include realization, disappointment, annoyance and approval. Grief, relief and pride have very few test examples, so their scores are unreliable.

## External checks
__EXT__

## Limitations and bias
- Trained on Reddit comments; scores drop on other kinds of text, such as long first-person reports.
- Labels come from human raters who often disagree on subtle emotions.
- English only. Reddit users are not representative of the general population, and the model may reflect those biases.
- A short text with no clear emotion may get no label above the threshold.

## How to use
```python
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo = "__REPO__"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForSequenceClassification.from_pretrained(repo)
inputs = tok("I can't believe I finally passed!", return_tensors="pt", truncation=True, max_length=64)
with torch.no_grad():
    probs = torch.sigmoid(model(**inputs).logits)[0]
threshold = __THRESHOLD__
print([(model.config.id2label[i], round(float(p), 3)) for i, p in enumerate(probs) if p >= threshold])
```
"""
repl = {
    "__THRESHOLD__": str(THRESHOLD), "__REPO__": REPO_ID, "__EXT__": ext,
    "__M28__": f(views["28 labels"]["macro_f1"]), "__U28__": f(views["28 labels"]["micro_f1"]),
    "__MEK__": f(views["Ekman 6 + neutral"]["macro_f1"]), "__UEK__": f(views["Ekman 6 + neutral"]["micro_f1"]),
    "__MHL__": f(views["12 headline emotions"]["macro_f1"]), "__UHL__": f(views["12 headline emotions"]["micro_f1"]),
}
for k, v in repl.items():
    card = card.replace(k, v)
open(f"{MODEL_DIR}/README.md", "w", encoding="utf-8").write(card)

api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(folder_path=MODEL_DIR, repo_id=REPO_ID, repo_type="model",
                  commit_message="MoodMentor BERT GoEmotions multi-label model")
print("Uploaded (private):", f"https://huggingface.co/{REPO_ID}")

# Check that the uploaded copy loads and runs
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(REPO_ID)
mdl = AutoModelForSequenceClassification.from_pretrained(REPO_ID)
enc = tok("I can't believe I finally passed!", return_tensors="pt", truncation=True, max_length=64)
with torch.no_grad():
    probs = torch.sigmoid(mdl(**enc).logits)[0]
top = probs.argsort(descending=True)[:3]
print("Top 3 from the uploaded model:", [(mdl.config.id2label[int(i)], round(float(probs[i]), 3)) for i in top])

In [ ]:
import torch
from huggingface_hub import whoami
from transformers import AutoTokenizer, AutoModelForSequenceClassification

REPO_ID = f"{whoami()['name']}/moodmentor-bert-goemotions"
tok = AutoTokenizer.from_pretrained(REPO_ID)
mdl = AutoModelForSequenceClassification.from_pretrained(REPO_ID).eval()
T = 0.7

tests = [
    "I'm so happy and proud of my little brother today!",
    "I miss her so much and I can't stop crying.",
    "This is the third time they cancelled on me, I'm furious.",
    "I'm excited about the trip but also really nervous.",
    "Thank you so much, you really helped me.",
    "I guess it's fine, whatever.",
    "I'm scared about my exam results tomorrow.",
    "That is disgusting, I can't even look at it.",
]
for s in tests:
    enc = tok(s, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        p = torch.sigmoid(mdl(**enc).logits)[0]
    found = [(mdl.config.id2label[i], round(float(v), 2)) for i, v in enumerate(p) if v >= T]
    top = p.argsort(descending=True)[:3]
    top3 = [(mdl.config.id2label[int(i)], round(float(p[i]), 2)) for i in top]
    print(s, "\n   above threshold:", found, "\n   top 3:", top3, "\n")

In [ ]:
!unzip -o -q moodmentor_colab_bundle.zip -d /content/moodmentor
%cd /content/moodmentor
!python -m ml.train_goemotions --model distilbert-base-uncased --name distilbert_repro
!python -m ml.score_external --name distilbert_repro --model-dir outputs/distilbert_repro_model

In [ ]:
from datasets import Dataset  # type: ignore

In [ ]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
!python -m ml.train_goemotions --model distilbert-base-uncased --name distilbert_goemotions

In [ ]:
!python -m ml.train_goemotions --model bert-base-uncased --name bert_goemotions